In [0]:
class InventoryAlertSystem:
    def __init__(self, api_client):
        self.api_client = api_client
        self.supplier_cache = {}

    def _get_supplier(self, product_id):
        if product_id in self.supplier_cache:
            return self.supplier_cache[product_id]
        try:
            supplier = self.api_client.get_supplier(product_id)
            self.supplier_cache[product_id] = supplier
            return supplier
        except Exception as e:
            print(f"Error fetching supplier for {product_id}: {e}")
            return None

    def _is_low_stock(self, item):
        return item.get("stock", 0) < item.get("reorder_threshold", 0)

    def get_alerts(self, warehouse_ids):
        result = []

        for warehouse_id in warehouse_ids:
            try:
                response = self.api_client.get_inventory(warehouse_id)
            except Exception as e:
                print(f"Error fetching inventory for {warehouse_id}: {e}")
                continue

            if not response or "inventory" not in response:
                continue

            for item in response.get("inventory", []):
                product_id = item.get("product_id")
                product_name = item.get("product_name")
                category = item.get("category")
                stock = item.get("stock")
                reorder_threshold = item.get("reorder_threshold")

                if any(v is None for v in [
                    product_id, product_name,
                    category, stock, reorder_threshold
                ]):
                    continue

                if not self._is_low_stock(item):
                    continue

                supplier = self._get_supplier(product_id)
                supplier_name = supplier.get("supplier_name") if supplier else "Unknown"
                lead_time_days = supplier.get("lead_time_days") if supplier else None

                result.append({
                    "product_id": product_id,
                    "product_name": product_name,
                    "category": category,
                    "stock": stock,
                    "reorder_threshold": reorder_threshold,
                    "supplier_name": supplier_name,
                    "lead_time_days": lead_time_days
                })

        result.sort(key=lambda x: x["stock"])
        return result


class MockAPIClient:
    INVENTORY = {
        "W1": {
            "warehouse_id": "W1",
            "inventory": [
                {"product_id": "P1", "product_name": "Laptop",  "category": "electronics", "stock": 5,  "reorder_threshold": 10},
                {"product_id": "P2", "product_name": "T-Shirt", "category": "clothing",    "stock": 2,  "reorder_threshold": 15},
                {"product_id": "P3", "product_name": "Rice",    "category": "grocery",     "stock": 50, "reorder_threshold": 20},
                {"product_id": "P4", "product_name": "Phone",   "category": "electronics", "stock": 0,  "reorder_threshold": 8},
                {"product_id": "P5", "product_name": "Jacket",  "category": "clothing",    "stock": 3,  "reorder_threshold": 10}
            ]
        }
    }

    SUPPLIERS = {
        "P1": {"product_id": "P1", "supplier_name": "TechCorp",  "lead_time_days": 3},
        "P2": {"product_id": "P2", "supplier_name": "FashionHub", "lead_time_days": 7},
        "P4": {"product_id": "P4", "supplier_name": "TechCorp",  "lead_time_days": 3},
        "P5": {"product_id": "P5", "supplier_name": "FashionHub", "lead_time_days": 7}
    }

    def get_inventory(self, warehouse_id):
        if warehouse_id not in self.INVENTORY:
            raise ValueError(f"No inventory for warehouse {warehouse_id}")
        return self.INVENTORY[warehouse_id]

    def get_supplier(self, product_id):
        if product_id not in self.SUPPLIERS:
            raise ValueError(f"No supplier for product {product_id}")
        return self.SUPPLIERS[product_id]


# Run it
api_client = MockAPIClient()
system = InventoryAlertSystem(api_client)
result = system.get_alerts(["W1"])
print(result)